# SBi-FAGAT v3 — Cloud GPU Benchmark Runner (Google Colab T4 / Kaggle GPU)

Offloads all remaining multi-seed GNN benchmarks (VN Temporal Holdout, VN Ablation & $K$-Curve, and optional Track 2 PHEME Tier 2A $K=10, N \ge 15$) from local CPU to **Google Colab T4** or **Kaggle T4/P100 GPU** in ~3-5 minutes.

### Quick Start
1. **Colab:** Select `Runtime -> Change runtime type -> T4 GPU`. Upload your local `data/processed/vn/` (`nodes.parquet`, `edges.parquet`, `nlp_features.pt`, and optional `data/processed/pheme/`) to Google Drive (e.g., `MyDrive/SBiFAGAT/data/processed/vn`), or mount Drive in Cell 2.
2. **Kaggle:** Enable `Accelerator -> GPU T4` and attach your processed dataset (`nodes.parquet`, `edges.parquet`, `nlp_features.pt`) under `/kaggle/input/...`.
3. Run all cells top-to-bottom. At the end, download `sbifagat_v3_gpu_results.zip` and extract into your local repo root before assembling the paper.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hieu-kiien/gnn-nlp-propagation.git"
REPO_BRANCH = os.environ.get("SBIF_BRANCH", "main")

is_kaggle = Path("/kaggle/working").exists()
work_root = Path("/kaggle/working/gnn-nlp-propagation") if is_kaggle else Path("/content/gnn-nlp-propagation")

if not Path("scripts/run_experiments.py").exists():
    if work_root.exists():
        os.chdir(work_root)
        subprocess.run(["git", "fetch", "--all"], check=True)
        subprocess.run(["git", "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
    else:
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(work_root)], check=True)
        os.chdir(work_root)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
for mod_name, pip_name in (
    ("torch_geometric", "torch-geometric"),
    ("umap", "umap-learn"),
    ("pydantic", "pydantic"),
    ("pyarrow", "pyarrow"),
    ("scipy", "scipy"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("tqdm", "tqdm"),
    ("pandas", "pandas"),
):
    try:
        __import__(mod_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

import torch
print("Working dir:", Path.cwd())
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))


In [ ]:
import shutil
import zipfile
from pathlib import Path

# Adjust GDRIVE_DATA_ROOT if your processed datasets live elsewhere in Drive
GDRIVE_DATA_ROOT = Path("/content/drive/MyDrive/SBiFAGAT/data/processed")
REQUIRED_FILES = ("nodes.parquet", "edges.parquet", "nlp_features.pt")
OPTIONAL_FILES = ("topic_splits.json",)
ALL_CORPUS_FILES = set(REQUIRED_FILES) | set(OPTIONAL_FILES)

for _corpus, _zip_p in (("vn", Path("/content/sbifagat_vn.zip")), ("pheme", Path("/content/sbifagat_pheme.zip"))):
    _parts = sorted(Path("/content").glob(f"{_zip_p.name}.part*"))
    if _parts and not _zip_p.exists():
        with open(_zip_p, "wb") as _out:
            for _part in _parts:
                _out.write(_part.read_bytes())
        for _part in _parts:
            _part.unlink(missing_ok=True)
        print(f"[{_corpus}] Reassembled {len(_parts)} parts into {_zip_p}")
    if _zip_p.is_file():
        _dst = Path("data/processed") / _corpus
        _dst.mkdir(parents=True, exist_ok=True)
        if not all((_dst / f).exists() for f in REQUIRED_FILES):
            with zipfile.ZipFile(_zip_p, "r") as _zf:
                for _member in _zf.infolist():
                    if _member.is_dir():
                        continue
                    _base = Path(_member.filename).name
                    if _base in ALL_CORPUS_FILES:
                        with _zf.open(_member, "r") as _src, open(_dst / _base, "wb") as _out:
                            shutil.copyfileobj(_src, _out)
            print(f"[{_corpus}] Extracted uploaded bundle from {_zip_p} into {_dst}")

IS_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = False
if not IS_KAGGLE:
    try:
        from google.colab import drive  # type: ignore
        IN_COLAB = True
        if (
            not Path("data/processed/vn/nlp_features.pt").exists()
            and not Path("/content/sbifagat_vn.zip").exists()
            and not Path("/content/drive/MyDrive").exists()
        ):
            drive.mount("/content/drive")
    except (ImportError, NotImplementedError, RuntimeError):
        IN_COLAB = False


def _stage_corpus(corpus: str) -> bool:
    dst_dir = Path("data/processed") / corpus
    dst_dir.mkdir(parents=True, exist_ok=True)
    if all((dst_dir / f).exists() for f in REQUIRED_FILES):
        print(f"[{corpus}] Already staged at {dst_dir}")
        return True

    search_roots = [GDRIVE_DATA_ROOT / corpus, Path("/content"), Path("/kaggle/input")]
    found: dict[str, Path] = {}
    nlp_candidates: list[Path] = []
    for root in search_roots:
        if not root.exists():
            continue
        for p in sorted(root.rglob("nlp_features.pt")):
            if not p.is_file():
                continue
            parts_low = [part.lower() for part in p.parts]
            if root == (GDRIVE_DATA_ROOT / corpus) or corpus in parts_low or any(corpus in part for part in parts_low[-3:-1]):
                nlp_candidates.append(p)
    for nlp_p in nlp_candidates:
        src_dir = nlp_p.parent
        found = {"nlp_features.pt": nlp_p}
        for fname in ALL_CORPUS_FILES:
            cand = src_dir / fname
            if cand.is_file():
                found[fname] = cand
        if all(f in found for f in REQUIRED_FILES):
            break
    if not all(f in found for f in REQUIRED_FILES) and nlp_candidates:
        for root in search_roots:
            if not root.exists():
                continue
            for p in sorted(root.rglob("*")):
                if not p.is_file() or p.name not in ALL_CORPUS_FILES:
                    continue
                parts_low = [part.lower() for part in p.parts]
                if root == (GDRIVE_DATA_ROOT / corpus) or corpus in parts_low or any(corpus in part for part in parts_low[-3:-1]):
                    found.setdefault(p.name, p)
    if all(f in found for f in REQUIRED_FILES):
        for fname, src_p in found.items():
            tgt_p = dst_dir / fname
            if not tgt_p.exists():
                try:
                    tgt_p.symlink_to(src_p)
                except OSError:
                    shutil.copy2(src_p, tgt_p)
        print(f"[{corpus}] Linked/staged from {found['nlp_features.pt'].parent}")
        return True

    print(f"[{corpus}] Not found locally; copy {REQUIRED_FILES} into {dst_dir.resolve()} if running this track.")
    return False


vn_ready = _stage_corpus("vn")
pheme_ready = _stage_corpus("pheme")
print("Dataset availability -> VN:", vn_ready, "| PHEME:", pheme_ready)


In [ ]:
import subprocess
import sys
from pathlib import Path

# Set True if you also want to re-produce checkpoints/best_sbifagat_vn.pt and main VN figures on GPU
RERUN_VN_CANONICAL = False


def _run(mod: str, *args: str) -> None:
    cmd = [sys.executable, "-u", "-m", mod, *args]
    print(">>>", " ".join(cmd))
    subprocess.run(cmd, check=True)


if Path("data/processed/vn/nlp_features.pt").exists():
    if RERUN_VN_CANONICAL:
        _run(
            "scripts.run_experiments",
            "--data-dir", "data/processed/vn",
            "--k-horizon", "20",
            "--hidden-dim", "64",
            "--edge-dim", "2",
            "--use-cascade-stats",
            "--min-total-replies", "25",
            "--polar-target-mode", "future_only",
            "--epochs", "100",
            "--patience", "10",
        )

    _run(
        "scripts.run_experiments",
        "--data-dir", "data/processed/vn",
        "--k-horizon", "20",
        "--hidden-dim", "64",
        "--edge-dim", "2",
        "--use-cascade-stats",
        "--min-total-replies", "25",
        "--polar-target-mode", "future_only",
        "--split-mode", "temporal",
        "--edge-source", "teacher",
        "--epochs", "100",
        "--patience", "10",
        "--skip-figures",
    )

    _run(
        "scripts.run_ablation",
        "--data-dir", "data/processed/vn",
        "--k-horizon", "20",
        "--hidden-dim", "64",
        "--edge-dim", "2",
        "--use-cascade-stats",
        "--min-total-replies", "25",
        "--polar-target-mode", "future_only",
        "--epochs", "100",
        "--patience", "10",
    )

    vn_ckpt = Path("checkpoints/best_sbifagat_vn.pt")
    if vn_ckpt.exists():
        _run(
            "scripts.analyze_mechanisms",
            "--data-dir", "data/processed/vn",
            "--ckpt", str(vn_ckpt),
            "--k-horizon", "20",
            "--min-total-replies", "25",
            "--polar-target-mode", "future_only",
        )
else:
    print("Skipping VN benchmarks: data/processed/vn/nlp_features.pt not found.")


In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

pheme_feat_path = Path("data/processed/pheme/nlp_features.pt")
if pheme_feat_path.exists():
    _pheme_payload = torch.load(pheme_feat_path, map_location="cpu", weights_only=False)
    _meta = _pheme_payload.get("_meta", {}) or {}
    _extracted_min_replies = int(
        _pheme_payload.get("min_total_replies_extracted", _meta.get("min_total_replies_extracted", 15))
    )
    del _pheme_payload
    if _extracted_min_replies > 15:
        print(
            f"Skipping optional Track 2 PHEME Tier 2A (min_total_replies=15): "
            f"attached pheme/nlp_features.pt was extracted at min_total_replies_extracted={_extracted_min_replies} (legacy Cohort B)."
        )
    else:
        subprocess.run([
            sys.executable, "-u", "-m", "scripts.export_dataset_stats",
            "--data-dir", "data/processed/pheme",
            "--names", "pheme",
            "--k-horizon", "10",
            "--min-total-replies", "15",
            "--polar-target-mode", "future_only",
            "--output-table", "paper/tables_dataset_eda_pheme.tex",
            "--output-json", "paper/results_dataset_eda_pheme.json",
        ], check=True)
        subprocess.run([
            sys.executable, "-u", "-m", "scripts.run_experiments",
            "--data-dir", "data/processed/pheme",
            "--k-horizon", "10",
            "--hidden-dim", "64",
            "--edge-dim", "2",
            "--use-cascade-stats",
            "--min-total-replies", "15",
            "--polar-target-mode", "future_only",
            "--epochs", "100",
            "--patience", "10",
            "--output-table", "paper/tables_main_pheme.tex",
            "--output-json", "paper/results_main_pheme.json",
        ], check=True)
        subprocess.run([
            sys.executable, "-u", "-m", "scripts.run_experiments",
            "--data-dir", "data/processed/pheme",
            "--k-horizon", "10",
            "--hidden-dim", "64",
            "--edge-dim", "2",
            "--use-cascade-stats",
            "--min-total-replies", "15",
            "--polar-target-mode", "future_only",
            "--eval-protocol", "loo",
            "--edge-source", "teacher",
            "--epochs", "100",
            "--patience", "10",
            "--output-table", "paper/tables_loo_pheme.tex",
            "--output-json", "paper/results_loo_pheme.json",
            "--skip-figures",
        ], check=True)
        subprocess.run([
            sys.executable, "-u", "-m", "scripts.run_ablation",
            "--data-dir", "data/processed/pheme",
            "--k-horizon", "10",
            "--hidden-dim", "64",
            "--edge-dim", "2",
            "--use-cascade-stats",
            "--min-total-replies", "15",
            "--polar-target-mode", "future_only",
            "--epochs", "100",
            "--patience", "10",
        ], check=True)
        pheme_ckpt = Path("checkpoints/best_sbifagat_pheme.pt")
        if pheme_ckpt.exists():
            subprocess.run([
                sys.executable, "-u", "-m", "scripts.analyze_mechanisms",
                "--data-dir", "data/processed/pheme",
                "--ckpt", str(pheme_ckpt),
                "--k-horizon", "10",
                "--min-total-replies", "15",
                "--polar-target-mode", "future_only",
            ], check=True)
else:
    print("Skipping optional Track 2 PHEME Tier 2A: data/processed/pheme/nlp_features.pt not found.")


In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

subprocess.run([
    sys.executable, "-u", "-m", "scripts.assemble_paper_artifacts",
    "--paper-dir", "paper",
], check=True)

BUNDLE_NAME = Path("/kaggle/working/sbifagat_v3_gpu_results.zip") if Path("/kaggle/working").exists() else Path("sbifagat_v3_gpu_results.zip")
FORBIDDEN_SUBSTRS = ("_machine_key", ".secret", ".env")
FORBIDDEN_EXTS = {".parquet", ".zip", ".bak"}

added = []
with zipfile.ZipFile(BUNDLE_NAME, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root_name in ("paper", "checkpoints"):
        root_dir = Path(root_name)
        if not root_dir.exists():
            continue
        for p in sorted(root_dir.rglob("*")):
            if not p.is_file():
                continue
            low = str(p).lower()
            if any(s in low for s in FORBIDDEN_SUBSTRS) or p.suffix.lower() in FORBIDDEN_EXTS:
                continue
            if p.name == "nlp_features.pt":
                continue
            zf.write(p, arcname=p.as_posix())
            added.append(p.as_posix())

print(f"Packaged {len(added)} result files into {BUNDLE_NAME.resolve()}")
content_zip = Path("/content/sbifagat_v3_gpu_results.zip")
if not Path("/kaggle/working").exists() and Path("/content").exists() and BUNDLE_NAME.resolve() != content_zip.resolve():
    shutil.copy2(BUNDLE_NAME, content_zip)
    print(f"Mirrored results bundle to {content_zip}")
if not Path("/kaggle/working").exists():
    try:
        from google.colab import files  # type: ignore
        files.download(str(BUNDLE_NAME))
    except (ImportError, NotImplementedError, RuntimeError):
        pass
